# KMeans clustering on ECG200 (shapelet features)

Goal: cluster ECG200 using **shapelet distance features** instead of the original 96 time points.

- One row = one ECG snippet (length 96)
- A shapelet = a short subsequence pattern (length $L$)
- One series $x$ becomes features $[d(x,s_1),\dots,d(x,s_m)]$
- Distance is the *best* match over all windows:

$$d(x,s)=\min_{t=0..T-L}\lVert x[t:t+L]-s\rVert_2$$

Note on normalization: ECG200 often looks per-series z-normalized already; **window z-normalization is optional**. In this notebook we compute distances both ways and compare.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Mitigates common MKL warning for KMeans on Windows
os.environ.setdefault('OMP_NUM_THREADS', '1')

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
    accuracy_score,
    normalized_mutual_info_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree


In [ ]:
# Load ECG200 (UCR TSV): col0=label, cols1..96=values
root_candidates = [
    (Path('..') / '..' / 'data' / 'UCRArchive_2018').resolve(),
    (Path('..') / '..' / 'data' / 'UCRArchive_2018' / 'UCRArchive_2018').resolve(),
]
UCR_ROOT = next((p for p in root_candidates if p.exists()), root_candidates[0])

dataset_name = 'ECG200'
train_path = UCR_ROOT / dataset_name / f'{dataset_name}_TRAIN.tsv'
test_path = UCR_ROOT / dataset_name / f'{dataset_name}_TEST.tsv'

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError(
        f'Could not find ECG200 files. Checked: {train_path} and {test_path}'
    )

train_df = pd.read_csv(train_path, sep='\t', header=None)
test_df = pd.read_csv(test_path, sep='\t', header=None)
df = pd.concat([train_df, test_df], ignore_index=True)

y_true = df.iloc[:, 0].to_numpy()
X = df.iloc[:, 1:].to_numpy()

print('UCR root:', UCR_ROOT)
print('Samples x TimePoints:', X.shape)
print('Classes:', np.unique(y_true))

row_means = X.mean(axis=1)
row_stds = X.std(axis=1, ddof=0)
print('Per-series mean (avg +/- std):', round(float(row_means.mean()), 4), '+/-', round(float(row_means.std()), 4))
print('Per-series std  (avg +/- std):', round(float(row_stds.mean()), 4), '+/-', round(float(row_stds.std()), 4))


In [ ]:
def z_norm_1d(a, eps=1e-8):
    a = np.asarray(a, dtype=float)
    mu = a.mean()
    sd = a.std(ddof=0)
    return (a - mu) / (sd + eps)


def min_distance_to_shapelet(ts, shapelet, z_norm_windows=True):
    ts = np.asarray(ts, dtype=float)
    s = np.asarray(shapelet, dtype=float)
    L = int(s.shape[0])
    if L > ts.shape[0]:
        raise ValueError('Shapelet longer than time series')

    windows = np.lib.stride_tricks.sliding_window_view(ts, window_shape=L)

    if z_norm_windows:
        w_mu = windows.mean(axis=1, keepdims=True)
        w_sd = windows.std(axis=1, ddof=0, keepdims=True)
        windows = (windows - w_mu) / (w_sd + 1e-8)
        s = z_norm_1d(s)

    d2 = ((windows - s) ** 2).sum(axis=1)
    best_idx = int(np.argmin(d2))
    best_dist = float(np.sqrt(d2[best_idx]))
    return best_dist, best_idx


def extract_random_shapelets(X, lengths=(15, 20, 25), n_shapelets=25, seed=42):
    rng = np.random.default_rng(seed)
    n_samples, T = X.shape

    shapelets = []
    for j in range(int(n_shapelets)):
        L = int(rng.choice(lengths))
        row = int(rng.integers(0, n_samples))
        start = int(rng.integers(0, T - L + 1))
        values = np.asarray(X[row, start:start + L], dtype=float).copy()
        shapelets.append(
            {
                'id': j,
                'length': L,
                'source_row': row,
                'start': start,
                'values': values,
            }
        )

    return shapelets


def shapelet_transform(X, shapelets, z_norm_windows=True):
    n_samples = X.shape[0]
    n_shapelets = len(shapelets)
    D = np.zeros((n_samples, n_shapelets), dtype=float)
    best_pos = np.zeros((n_samples, n_shapelets), dtype=int)

    for i in range(n_samples):
        ts = X[i]
        for j, sh in enumerate(shapelets):
            dist, pos = min_distance_to_shapelet(
                ts, sh['values'], z_norm_windows=z_norm_windows
            )
            D[i, j] = dist
            best_pos[i, j] = pos

    return D, best_pos


In [ ]:
# Build shapelets and compute shapelet-distance features
shapelets = extract_random_shapelets(X, lengths=(15, 20, 25), n_shapelets=25, seed=42)
shapelet_meta = pd.DataFrame(
    [
        {
            'shapelet_id': sh['id'],
            'length': sh['length'],
            'source_row': sh['source_row'],
            'start': sh['start'],
        }
        for sh in shapelets
    ]
).sort_values(['length', 'shapelet_id']).reset_index(drop=True)

# Distances with and without per-window z-normalization
D_winznorm, best_pos_winznorm = shapelet_transform(X, shapelets, z_norm_windows=True)
D_noznorm, best_pos_noznorm = shapelet_transform(X, shapelets, z_norm_windows=False)

print('Shapelet feature matrices:', D_winznorm.shape, D_noznorm.shape)
display(shapelet_meta.head(10))

# Two clustering inputs (both standardized so shapelets are on comparable scales)
views = {
    'shapelet_dist_winznorm_std': StandardScaler().fit_transform(D_winznorm),
    'shapelet_dist_noznorm_std': StandardScaler().fit_transform(D_noznorm),
}


## One shapelet across many windows

A single shapelet is compared to **all windows** of a series (same length) and we keep the **best match** (minimum distance). That one number becomes a feature for clustering.

In [ ]:
shapelet_id = 0
sh = shapelets[shapelet_id]
L = sh['length']

# Use the window-z-normalized distances for the demo
dists = D_winznorm[:, shapelet_id]
order = np.argsort(dists)

top_k = 3
print(f'Shapelet {shapelet_id}: length={L}, extracted from row={sh["source_row"]}, start={sh["start"]}')
print('Top matches (series index -> distance -> best window start):')
for rank in range(top_k):
    i = int(order[rank])
    pos = int(best_pos_winznorm[i, shapelet_id])
    print(f'  {i} -> {dists[i]:.4f} -> start={pos}')

fig, axes = plt.subplots(top_k, 2, figsize=(12, 3 * top_k))
if top_k == 1:
    axes = np.array([axes])

for rank in range(top_k):
    i = int(order[rank])
    pos = int(best_pos_winznorm[i, shapelet_id])
    ts = X[i]

    ax_ts = axes[rank, 0]
    ax_ts.plot(ts, color='black', linewidth=1)
    ax_ts.axvspan(pos, pos + L - 1, color='tab:orange', alpha=0.25)
    ax_ts.set_title(f'Series {i} (best dist={dists[i]:.3f})')
    ax_ts.set_xlabel('time index')
    ax_ts.set_ylabel('value')
    ax_ts.grid(alpha=0.2)

    ax_sh = axes[rank, 1]
    ax_sh.plot(z_norm_1d(sh['values']), label='shapelet (z-norm)', color='tab:blue')
    window = ts[pos:pos + L]
    ax_sh.plot(z_norm_1d(window), label='matched window (z-norm)', color='tab:orange')
    ax_sh.set_title('Shapelet vs matched window')
    ax_sh.set_xlabel('within-window index')
    ax_sh.grid(alpha=0.2)
    ax_sh.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Unsupervised k selection (same approach as the original notebook)
k_values = list(range(2, 9))
seed_list = [0, 1, 2, 3, 4]

def elbow_k_from_line(k_list, inertia_list):
    x = np.asarray(k_list, dtype=float)
    y = np.asarray(inertia_list, dtype=float)
    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])
    line = p2 - p1
    line_norm = np.linalg.norm(line)
    if line_norm == 0:
        return int(k_list[0])
    distances = []
    for i in range(len(x)):
        p = np.array([x[i], y[i]])
        distances.append(np.abs(np.cross(line, p - p1)) / line_norm)
    return int(k_list[int(np.argmax(distances))])

rows = []
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (view_name, X_view) in enumerate(views.items()):
    inertias = []
    silhouettes = []
    silhouettes_std = []
    ch_scores = []
    ch_scores_std = []
    db_scores = []
    db_scores_std = []
    stabilities = []
    stabilities_std = []

    for k in k_values:
        labels_by_seed = []
        inertias_by_seed = []
        sil_by_seed = []
        ch_by_seed = []
        db_by_seed = []

        for seed in seed_list:
            km = KMeans(n_clusters=k, random_state=seed, n_init=20)
            labels = km.fit_predict(X_view)
            labels_by_seed.append(labels)
            inertias_by_seed.append(float(km.inertia_))
            sil_by_seed.append(float(silhouette_score(X_view, labels)))
            ch_by_seed.append(float(calinski_harabasz_score(X_view, labels)))
            db_by_seed.append(float(davies_bouldin_score(X_view, labels)))

        inertias.append(float(np.mean(inertias_by_seed)))
        silhouettes.append(float(np.mean(sil_by_seed)))
        silhouettes_std.append(float(np.std(sil_by_seed, ddof=0)))
        ch_scores.append(float(np.mean(ch_by_seed)))
        ch_scores_std.append(float(np.std(ch_by_seed, ddof=0)))
        db_scores.append(float(np.mean(db_by_seed)))
        db_scores_std.append(float(np.std(db_by_seed, ddof=0)))

        pairwise_ari = []
        for i in range(len(labels_by_seed)):
            for j in range(i + 1, len(labels_by_seed)):
                pairwise_ari.append(adjusted_rand_score(labels_by_seed[i], labels_by_seed[j]))
        stabilities.append(float(np.mean(pairwise_ari)))
        stabilities_std.append(float(np.std(pairwise_ari, ddof=0)))

    elbow_k = elbow_k_from_line(k_values, inertias)

    for i, k in enumerate(k_values):
        rows.append({
            'view': view_name,
            'k': k,
            'inertia': inertias[i],
            'silhouette': silhouettes[i],
            'silhouette_std': silhouettes_std[i],
            'calinski_harabasz': ch_scores[i],
            'calinski_harabasz_std': ch_scores_std[i],
            'davies_bouldin': db_scores[i],
            'davies_bouldin_std': db_scores_std[i],
            'stability_ari': stabilities[i],
            'stability_ari_std': stabilities_std[i],
            'elbow_k': elbow_k,
            'elbow_match': int(k == elbow_k),
        })

    axes[idx, 0].plot(k_values, inertias, marker='o')
    axes[idx, 0].axvline(elbow_k, color='r', linestyle='--', alpha=0.6)
    axes[idx, 0].set_title(f'{view_name}: elbow (mean inertia)')
    axes[idx, 0].set_xlabel('k')
    axes[idx, 0].set_ylabel('inertia')
    axes[idx, 0].grid(alpha=0.3)

    axes[idx, 1].plot(k_values, silhouettes, marker='o', color='tab:green')
    axes[idx, 1].set_title(f'{view_name}: silhouette (mean over seeds)')
    axes[idx, 1].set_xlabel('k')
    axes[idx, 1].set_ylabel('silhouette')
    axes[idx, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

selection_df = pd.DataFrame(rows)

for metric in ['silhouette', 'calinski_harabasz', 'stability_ari']:
    selection_df[f'rank_{metric}'] = selection_df.groupby('view')[metric].rank(method='dense', ascending=False)
selection_df['rank_davies_bouldin'] = selection_df.groupby('view')['davies_bouldin'].rank(method='dense', ascending=True)

selection_df['rank_sum'] = (
    selection_df['rank_silhouette']
    + selection_df['rank_calinski_harabasz']
    + selection_df['rank_davies_bouldin']
    + selection_df['rank_stability_ari']
    - 0.5 * selection_df['elbow_match']
)

min_k = min(k_values)
k_penalty = 0.15
selection_df['rank_sum_penalized'] = selection_df['rank_sum'] + k_penalty * (selection_df['k'] - min_k)

selection_df = selection_df.sort_values(
    ['view', 'rank_sum_penalized', 'silhouette', 'stability_ari', 'k'],
    ascending=[True, True, False, False, True],
).reset_index(drop=True)

display(selection_df[[
    'view',
    'k',
    'silhouette',
    'silhouette_std',
    'calinski_harabasz',
    'davies_bouldin',
    'stability_ari',
    'stability_ari_std',
    'elbow_k',
    'rank_sum',
    'rank_sum_penalized',
]])

best_k_by_view = {}
for view_name in views:
    top_row = selection_df[selection_df['view'] == view_name].iloc[0]
    best_k_by_view[view_name] = int(top_row['k'])

best_rows_df = selection_df.groupby('view', as_index=False).first()
final_row = best_rows_df.sort_values(
    ['rank_sum_penalized', 'silhouette', 'stability_ari'],
    ascending=[True, False, False],
).iloc[0]

final_view = str(final_row['view'])
final_k = int(final_row['k'])

print('Best k by view:', best_k_by_view)
print('Final unsupervised choice (no labels used):', {'view': final_view, 'k': final_k})

In [ ]:
# Fit KMeans using each view-specific best k
models_by_view = {}
labels_by_view = {}
summary_rows = []

for view_name, X_view in views.items():
    k_view = best_k_by_view[view_name]
    model = KMeans(n_clusters=k_view, random_state=42, n_init=50)
    labels_view = model.fit_predict(X_view)

    models_by_view[view_name] = model
    labels_by_view[view_name] = labels_view

    summary_rows.append({
        'view': view_name,
        'best_k': int(k_view),
        'inertia': float(model.inertia_),
        'silhouette': float(silhouette_score(X_view, labels_view)),
        'calinski_harabasz': float(calinski_harabasz_score(X_view, labels_view)),
        'davies_bouldin': float(davies_bouldin_score(X_view, labels_view)),
    })

fit_summary_df = pd.DataFrame(summary_rows).sort_values('view').reset_index(drop=True)
display(fit_summary_df)

final_model = models_by_view[final_view]
final_labels = labels_by_view[final_view]
print('Final unsupervised model:', {'view': final_view, 'k': int(final_model.n_clusters)})


In [ ]:
# Post-hoc label benchmark (uses y_true only for evaluation)
def majority_label_mapping(y_ref, clusters):
    mapping = {}
    for c in np.unique(clusters):
        mapping[c] = pd.Series(y_ref[clusters == c]).mode().iloc[0]
    return mapping

def mapped_predictions(clusters, mapping):
    return np.array([mapping[c] for c in clusters])

benchmark_rows = []

for view_name, X_view in views.items():
    k_candidates = sorted(set([2, best_k_by_view[view_name]]))
    for k_eval in k_candidates:
        km_eval = KMeans(n_clusters=k_eval, random_state=42, n_init=50)
        cl_eval = km_eval.fit_predict(X_view)
        map_eval = majority_label_mapping(y_true, cl_eval)
        pred_eval = mapped_predictions(cl_eval, map_eval)

        benchmark_rows.append({
            'view': view_name,
            'k': int(k_eval),
            'mapped_accuracy': float(accuracy_score(y_true, pred_eval)),
            'ari': float(adjusted_rand_score(y_true, cl_eval)),
            'nmi': float(normalized_mutual_info_score(y_true, cl_eval)),
            'silhouette': float(silhouette_score(X_view, cl_eval)),
            'clusters': int(len(np.unique(cl_eval))),
            'classes': int(len(np.unique(y_true))),
        })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(['view', 'k']).reset_index(drop=True)
display(benchmark_df)

print('Note: the table above uses ground-truth labels for evaluation only (not for selecting k/view).')
print('Unsupervised choice (no labels used):', {'view': final_view, 'k': final_k})


In [ ]:
# PCA plots in shapelet-feature space (both views with their best k)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

view_order = list(views.keys())
for row_idx, view_name in enumerate(view_order):
    X_view = views[view_name]
    clusters = labels_by_view[view_name]
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_view)

    axes[row_idx, 0].scatter(X_2d[:, 0], X_2d[:, 1], c=clusters, s=35, cmap='tab10', alpha=0.8)
    axes[row_idx, 0].set_title(f'{view_name}: PCA by cluster (k={best_k_by_view[view_name]})')
    axes[row_idx, 0].set_xlabel('PC1')
    axes[row_idx, 0].set_ylabel('PC2')
    axes[row_idx, 0].grid(alpha=0.2)

    axes[row_idx, 1].scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, s=35, cmap='coolwarm', alpha=0.8)
    axes[row_idx, 1].set_title(f'{view_name}: PCA by true label')
    axes[row_idx, 1].set_xlabel('PC1')
    axes[row_idx, 1].set_ylabel('PC2')
    axes[row_idx, 1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## Functional boxplot (pattern summary)

This summarizes a set of curves by:
- a **median curve** (deepest)
- a **central envelope** (deepest 50%)
- optional **outliers** (curves leaving the inflated envelope)

We compute it on the **original ECG curves `X`**, but grouped by the clusters learned from shapelet features.

In [ ]:
def functional_boxplot_stats(curves, central_frac=0.5, factor=1.5):
    curves = np.asarray(curves, dtype=float)
    if curves.ndim != 2:
        raise ValueError('curves must have shape (n_curves, n_timepoints)')

    n, T = curves.shape
    if n < 3:
        raise ValueError('need at least 3 curves')

    # Modified band depth for k=2 computed from ranks at each time point.
    # At time t, if a curve has rank r (1..n), #bands containing it is (r-1)*(n-r).
    ranks = np.argsort(np.argsort(curves, axis=0), axis=0) + 1  # (n, T), 1..n
    denom = (n * (n - 1)) / 2.0
    depth_t = ((ranks - 1) * (n - ranks)) / denom
    depth = depth_t.mean(axis=1)  # (n,)

    order = np.argsort(-depth)  # deepest first
    n_central = max(1, int(np.ceil(float(central_frac) * n)))
    central_idx = order[:n_central]

    central = curves[central_idx]
    median = curves[order[0]]

    lower = central.min(axis=0)
    upper = central.max(axis=0)
    spread = upper - lower

    lower_fence = lower - factor * spread
    upper_fence = upper + factor * spread

    outlier_mask = (curves < lower_fence).any(axis=1) | (curves > upper_fence).any(axis=1)

    return {
        'depth': depth,
        'order': order,
        'median': median,
        'lower': lower,
        'upper': upper,
        'lower_fence': lower_fence,
        'upper_fence': upper_fence,
        'outlier_mask': outlier_mask,
        'central_idx': central_idx,
    }


def plot_functional_boxplot(curves, title, ax=None, show_outliers=True):
    curves = np.asarray(curves, dtype=float)
    stats = functional_boxplot_stats(curves)
    t = np.arange(curves.shape[1])

    if ax is None:
        ax = plt.gca()

    # Background curves
    ax.plot(t, curves.T, color='0.7', alpha=0.25, linewidth=0.7)

    # Central region
    ax.fill_between(t, stats['lower'], stats['upper'], color='tab:blue', alpha=0.20, label='central (50%)')

    # Median
    ax.plot(t, stats['median'], color='tab:red', linewidth=2, label='median')

    # Outliers
    if show_outliers and stats['outlier_mask'].any():
        out = curves[stats['outlier_mask']]
        ax.plot(t, out.T, color='tab:orange', alpha=0.65, linewidth=1.0, label='outliers')

    ax.set_title(title)
    ax.set_xlabel('time index')
    ax.grid(alpha=0.2)
    return stats


# Overall (all series)
plt.figure(figsize=(12, 4))
plot_functional_boxplot(X, title='All ECG200 series (functional boxplot)')
plt.tight_layout()
plt.show()

# Per cluster using the final unsupervised model (clusters were learned in shapelet space)
clusters = np.asarray(final_labels)
uniq = np.unique(clusters)

ncols = 2
nrows = int(np.ceil(len(uniq) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.5 * nrows), squeeze=False)

for idx, c in enumerate(uniq):
    r = idx // ncols
    cc = idx % ncols
    curves_c = X[clusters == c]
    plot_functional_boxplot(curves_c, title=f'Cluster {int(c)} (n={curves_c.shape[0]})', ax=axes[r, cc])

# Hide any unused axes
for idx in range(len(uniq), nrows * ncols):
    r = idx // ncols
    cc = idx % ncols
    axes[r, cc].axis('off')

plt.tight_layout()
plt.show()


## Shapelet patterns per cluster

In shapelet space, each feature is a **distance to one shapelet** (smaller = shapelet appears somewhere). Below: shapelets with the **smallest average distance** inside each cluster.

In [ ]:
# Use un-standardized distances for interpretation (standardization is for KMeans geometry)
D_by_view = {
    'shapelet_dist_winznorm_std': D_winznorm,
    'shapelet_dist_noznorm_std': D_noznorm,
}
D_final = D_by_view[final_view]

shapelet_lengths = np.array([sh['length'] for sh in shapelets], dtype=float)
len_sqrt = np.sqrt(shapelet_lengths)

clusters = np.unique(final_labels)
top_n = 5

rows = []
for c in clusters:
    idx = np.where(final_labels == c)[0]
    mean_dist = D_final[idx].mean(axis=0)

    # Compare different lengths using RMS distance: dist / sqrt(L)
    mean_rms = mean_dist / len_sqrt
    top = np.argsort(mean_rms)[:top_n]

    for rank, j in enumerate(top, start=1):
        rows.append(
            {
                'cluster': int(c),
                'rank': int(rank),
                'shapelet_id': int(shapelets[j]['id']),
                'length': int(shapelets[j]['length']),
                'mean_distance': float(mean_dist[j]),
                'mean_rms_distance': float(mean_rms[j]),
            }
        )

top_shapelets_df = (
    pd.DataFrame(rows)
    .sort_values(['cluster', 'rank'])
    .reset_index(drop=True)
)

display(top_shapelets_df)

# Plot the selected shapelets
n_rows = len(clusters)
fig, axes = plt.subplots(n_rows, top_n, figsize=(3.2 * top_n, 2.8 * n_rows), squeeze=False)

for r, c in enumerate(clusters):
    c_rows = top_shapelets_df[top_shapelets_df['cluster'] == int(c)]
    for col, (_, row) in enumerate(c_rows.iterrows()):
        sh_id = int(row['shapelet_id'])
        sh = shapelets[sh_id]
        ax = axes[r, col]
        ax.plot(z_norm_1d(sh['values']), color='tab:blue', linewidth=2)
        ax.set_title(f"c{int(c)} #{int(row['rank'])}: sh{sh_id} (L={sh['length']})")
        ax.grid(alpha=0.2)

for ax in axes.ravel():
    ax.set_xticks([])

plt.tight_layout()
plt.show()


In [ ]:
# Surrogate decision trees to explain KMeans in shapelet-feature space
# Rules are about distance-to-shapelet features: smaller distance => that shapelet pattern appears somewhere.

feature_names = [f'shapelet_{sh["id"]}_L{sh["length"]}' for sh in shapelets]

surrogates_by_view = {}
for view_name, X_used in views.items():
    cluster_labels = labels_by_view[view_name]

    surrogate = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8, random_state=42)
    surrogate.fit(X_used, cluster_labels)

    train_fidelity = accuracy_score(cluster_labels, surrogate.predict(X_used))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_fidelity = cross_val_score(surrogate, X_used, cluster_labels, cv=cv, scoring='accuracy').mean()

    print(f'\nView: {view_name}')
    print('Surrogate tree train fidelity:', round(float(train_fidelity), 4))
    print('Surrogate tree CV fidelity:', round(float(cv_fidelity), 4))

    print('\nSurrogate rules:\n')
    print(export_text(surrogate, feature_names=feature_names))

    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': surrogate.feature_importances_,
    }).sort_values('importance', ascending=False).reset_index(drop=True)
    display(importance_df.head(10))

    plt.figure(figsize=(16, 7))
    plot_tree(
        surrogate,
        feature_names=feature_names,
        class_names=[str(c) for c in sorted(np.unique(cluster_labels))],
        filled=True,
        rounded=True,
        impurity=False,
        fontsize=7,
    )
    plt.title(f'Decision Tree Surrogate for KMeans Clusters ({view_name})')
    plt.show()

    surrogates_by_view[view_name] = {
        'surrogate': surrogate,
        'X_used': X_used,
        'feature_names': feature_names,
    }


In [ ]:
# SHAP feature attributions for surrogate trees (optional)
try:
    import shap
    shap_available = True
except ImportError:
    shap_available = False
    print('SHAP is not installed in this environment. Install with: pip install shap')

if shap_available:
    for view_name, payload in surrogates_by_view.items():
        surrogate = payload['surrogate']
        X_used = payload['X_used']
        feature_names = payload['feature_names']

        explainer = shap.TreeExplainer(surrogate)
        raw_shap = explainer.shap_values(X_used)

        if isinstance(raw_shap, list):
            shap_abs_mean = np.mean(
                np.stack([np.abs(class_values).mean(axis=0) for class_values in raw_shap], axis=0),
                axis=0,
            )
            shap_for_beeswarm = raw_shap[0]
        elif isinstance(raw_shap, np.ndarray) and raw_shap.ndim == 3:
            shap_abs_mean = np.abs(raw_shap).mean(axis=(0, 2))
            shap_for_beeswarm = raw_shap[:, :, 0]
        else:
            shap_abs_mean = np.abs(raw_shap).mean(axis=0)
            shap_for_beeswarm = raw_shap

        shap_importance_df = pd.DataFrame({
            'feature': feature_names,
            'mean_abs_shap': shap_abs_mean,
        }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

        print(f'\nView: {view_name}')
        display(shap_importance_df.head(15))

        top_n = 15
        top_df = shap_importance_df.head(top_n).iloc[::-1]
        plt.figure(figsize=(8, 6))
        plt.barh(top_df['feature'], top_df['mean_abs_shap'])
        plt.title(f'Top SHAP Features for KMeans Surrogate ({view_name})')
        plt.xlabel('Mean(|SHAP value|)')
        plt.tight_layout()
        plt.show()

        try:
            shap.summary_plot(
                shap_for_beeswarm,
                pd.DataFrame(X_used, columns=feature_names),
                max_display=15,
                show=True,
            )
        except Exception as ex:
            print('Could not render SHAP beeswarm summary:', ex)
